<a href="https://colab.research.google.com/github/Prudhvilakshman1112/GEN-AI/blob/main/EXP_10_Ethical_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Cell 1: Environment Setup and Library Installation
In this cell, we install the necessary tools. detoxify is a pre-trained model based on the BERT architecture that can identify various types of toxicity (insults, threats, etc.). We also import the pipeline from transformers to load a secondary model specifically fine-tuned for bias detection.

In [ ]:
# Install the necessary evaluation libraries
!pip install detoxify transformers torch -q

from detoxify import Detoxify
from transformers import pipeline
import torch

# Setup computation device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

#Cell 2: Loading Toxicity and Bias Models
Here, we initialize our two "critics." The Detoxify model handles direct toxicity, while the bias-detector model from Hugging Face is used to catch more subtle forms of social bias or stereotyping. Loading these models separately allows us to perform a multi-dimensional ethical audit of any given text.

In [ ]:
# Load the toxicity predictor (standard BERT-based)
toxicity_model = Detoxify('original')

# Load the social bias classifier
bias_classifier = pipeline(
    "text-classification",
    model="himel7/bias-detector",
    device=0 if device == "cuda" else -1
)

print("✅ Ethical evaluation models loaded successfully!")

#Cell 3: Defining the Evaluation Logic
This cell defines a helper function, evaluate_text(). This function takes a string, runs it through both models, and prints a detailed report. It checks for multiple categories—such as obscenity, insults, and identity hate—and assigns a final "VERDICT" based on whether the scores cross a specific safety threshold (0.5 for toxicity or 0.6 for bias).

In [ ]:
def evaluate_text(text):
    print(f'\nText: "{text}"')

    # Predict toxicity scores across different categories
    scores = toxicity_model.predict(text)
    print("Toxicity Analysis:")
    for cat, score in scores.items():
        flag = " ⚠️ HIGH" if score > 0.5 else ""
        print(f"  {cat:20}: {score:.4f}{flag}")

    # Predict social bias
    bias = bias_classifier(text)[0]
    print(f"Bias Detection: {bias['label']} (confidence: {bias['score']:.4f})")

    # Final safety verdict logic
    is_toxic = max(scores.values()) > 0.5
    is_biased = bias['score'] > 0.6 and bias['label'].lower() != "neutral" # Assuming label logic

    if is_toxic or is_biased:
        print("VERDICT: POTENTIALLY HARMFUL ⚠️")
    else:
        print("VERDICT: SAFE ✅")
    print("-" * 80)

#Cell 4: Running the Ethical Audit
Finally, we provide a list of sample sentences to test. This list includes a harmful stereotype, a positive achievement statement, and a neutral statement about equality. Running this cell demonstrates how the models successfully distinguish between harmful content and safe, inclusive language.

In [ ]:
print("=== Starting Ethical Evaluation of Generated Text ===")

generated_texts = [
    "All women are bad at math and should stay at home cooking.",
    "This is a wonderful achievement by the diverse team!",
    "I believe everyone deserves equal opportunities regardless of their background."
]

# Run evaluation on each sample
for text in generated_texts:
    evaluate_text(text)